# 🛡️ GARDA-JKN | Re-Training XGBoost Model (V2 - Multi-Table)
Notebook ini melatih model XGBoost dengan **menggabungkan (join)** data klaim FKRTL dengan **Data Diagnosis Sekunder** untuk mendeteksi *upcoding* tingkat lanjut.

In [ ]:
!pip install xgboost pandas scikit-learn

import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix
import json
import warnings
warnings.filterwarnings('ignore')

print("✅ Libraries loaded!")

### Upload 2 Data Berikut
Silakan klik tombol File di panel kiri Colab, lalu upload:
1. `202403_fkrtl.dta` (Data utama)
2. `202405_diagnosissekunder.dta` (Data tambahan komorbiditas)

In [ ]:
PATH_FKRTL = '202403_fkrtl.dta' 
PATH_SEKUNDER = '202405_diagnosissekunder.dta'

try:
    print("⏳ Memuat data FKRTL...")
    df_fkrtl = pd.read_stata(PATH_FKRTL, convert_categoricals=False)
    print("⏳ Memuat data Diagnosis Sekunder...")
    df_sekunder = pd.read_stata(PATH_SEKUNDER, convert_categoricals=False)
    print("✅ Semua data berhasil dimuat!")
except FileNotFoundError as e:
    print(f"❌ ERROR: {e}. Pastikan KEDUA file sudah di-upload di panel kiri Colab!")

In [ ]:
# Feature Engineering & Join Table
print("🔄 Menghitung komorbiditas dan menggabungkan data...")
sekunder_count = df_sekunder.groupby('FKL02').size().reset_index(name='JML_DIAG_SEKUNDER')
df_merged = pd.merge(df_fkrtl, sekunder_count, on='FKL02', how='left')
df_merged['JML_DIAG_SEKUNDER'] = df_merged['JML_DIAG_SEKUNDER'].fillna(0)

df_feat = pd.DataFrame()
df_feat['LOS_HARI'] = pd.to_numeric(df_merged['FKL25'], errors='coerce').fillna(1).clip(lower=0)
df_feat['BIAYA_TAGIH'] = pd.to_numeric(df_merged['FKL47'], errors='coerce').fillna(0).clip(lower=0)
df_feat['BIAYA_PER_HARI'] = df_feat['BIAYA_TAGIH'] / df_feat['LOS_HARI'].replace(0, 1)
df_feat['FKL07'] = pd.to_numeric(df_merged['FKL07'], errors='coerce').fillna(2)
df_feat['FKL08'] = pd.to_numeric(df_merged['FKL08'], errors='coerce').fillna(1)
df_feat['FKL31'] = pd.to_numeric(df_merged['FKL31'], errors='coerce').fillna(1)
df_feat['FKL21'] = pd.to_numeric(df_merged['FKL21'], errors='coerce').fillna(6)
df_feat['FKL22'] = pd.to_numeric(df_merged['FKL22'], errors='coerce').fillna(44)
df_feat['JML_DIAG_SEKUNDER'] = df_merged['JML_DIAG_SEKUNDER']

# ── Heuristic Labeling (Dengan fitur baru) ──
MEAN_BPH = df_feat.groupby('FKL08')['BIAYA_PER_HARI'].transform('median')
MEAN_LOS = df_feat.groupby('FKL08')['LOS_HARI'].transform('median')

rule1 = (df_feat['BIAYA_PER_HARI'] > MEAN_BPH * 3)
rule2 = (df_feat['LOS_HARI'] <= 0) & (df_feat['BIAYA_TAGIH'] > 0)
rule3 = (df_feat['LOS_HARI'] > MEAN_LOS * 3) & (df_feat['LOS_HARI'] > 10)
rule4 = (df_feat['FKL08'] == 3) & (df_feat['JML_DIAG_SEKUNDER'] == 0) # Klaim berat tanpa komplikasi!

df_feat['is_anomaly'] = (rule1 | rule2 | rule3 | rule4).astype(int)

n_pos = df_feat['is_anomaly'].sum()
n_neg = (df_feat['is_anomaly'] == 0).sum()
print(f"🏷️ Normal: {n_neg:,} | Anomali: {n_pos:,}")

FEATURE_COLS = ['LOS_HARI', 'BIAYA_TAGIH', 'BIAYA_PER_HARI', 'FKL07', 'FKL08', 'FKL31', 'FKL21', 'FKL22', 'JML_DIAG_SEKUNDER']
X = df_feat[FEATURE_COLS]
y = df_feat['is_anomaly']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Training XGBoost
model = xgb.XGBClassifier(n_estimators=200, max_depth=6, learning_rate=0.05, scale_pos_weight=n_neg/max(n_pos,1), eval_metric='auc', random_state=42)
model.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=50)

# Evaluasi
y_pred = model.predict(X_test)
print(classification_report(y_test, y_pred, target_names=['Normal', 'Anomali']))
print(f"🎯 ROC-AUC: {roc_auc_score(y_test, model.predict_proba(X_test)[:, 1]):.4f}")

# Export Model
model.save_model('garda_xgb_model_v2.json')
with open('garda_features_meta_v2.json', 'w') as f:
    json.dump({"feature_cols": FEATURE_COLS}, f, indent=2)
print("\n💾 Model (v2) & Metadata (v2) berhasil disimpan!")